In [1]:
from collections import defaultdict
import os
import torch
import random
import numpy as np
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def ReadLineFromFile(path):
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        yield eval(l)
        
'''
Set seeds
'''
seed = 2022
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

In [2]:
# return (user item timestamp) sort in get_interaction
def Amazon(dataset_name, rating_score):
    '''
    reviewerID - ID of the reviewer, e.g. A2SUAM1J3GNN3B
    asin - ID of the product, e.g. 0000013714
    reviewerName - name of the reviewer
    helpful - helpfulness rating of the review, e.g. 2/3
    --"helpful": [2, 3],
    reviewText - text of the review
    --"reviewText": "I bought this for my husband who plays the piano. ..."
    overall - rating of the product
    --"overall": 5.0,
    summary - summary of the review
    --"summary": "Heavenly Highway Hymns",
    unixReviewTime - time of the review (unix time)
    --"unixReviewTime": 1252800000,
    reviewTime - time of the review (raw)
    --"reviewTime": "09 13, 2009"
    '''
    datas = []
    # older Amazon
    data_file = '../data/'+dataset_name+'/reviews_' + dataset_name + '_5.json.gz'
    # latest Amazon
    # data_file = '/home/hui_wang/data/new_Amazon/' + dataset_name + '.json.gz'
    for inter in parse(data_file):
        if float(inter['overall']) <= rating_score: # 小于一定分数去掉
            continue
        user = inter['reviewerID']
        item = inter['asin']
        time = inter['unixReviewTime']
        rev = inter['reviewText']
        datas.append((user, item,rev , int(time)))
    return datas

def Amazon_meta(dataset_name, data_maps):
    '''
    asin - ID of the product, e.g. 0000031852
    --"asin": "0000031852",
    title - name of the product
    --"title": "Girls Ballet Tutu Zebra Hot Pink",
    description
    price - price in US dollars (at time of crawl)
    --"price": 3.17,
    imUrl - url of the product image (str)
    --"imUrl": "http://ecx.images-amazon.com/images/I/51fAmVkTbyL._SY300_.jpg",
    related - related products (also bought, also viewed, bought together, buy after viewing)
    --"related":{
        "also_bought": ["B00JHONN1S"],
        "also_viewed": ["B002BZX8Z6"],
        "bought_together": ["B002BZX8Z6"]
    },
    salesRank - sales rank information
    --"salesRank": {"Toys & Games": 211836}
    brand - brand name
    --"brand": "Coxlures",
    categories - list of categories the product belongs to
    --"categories": [["Sports & Outdoors", "Other Sports", "Dance"]]
    '''
    datas = {}
    meta_file = '../data/'+dataset_name+'/meta_' + dataset_name + '.json.gz'
    item_asins = list(data_maps['item2id'].keys())
    for info in parse(meta_file):
        if info['asin'] not in item_asins:
            continue
        datas[info['asin']] = info
    return datas

def add_comma(num):
    # 1000000 -> 1,000,000
    str_num = str(num)
    res_num = ''
    for i in range(len(str_num)):
        res_num += str_num[i]
        if (len(str_num)-i-1) % 3 == 0:
            res_num += ','
    return res_num[:-1]

# categories 和 brand is all attribute
def get_attribute_Amazon(meta_infos, datamaps, attribute_core):

    attributes = defaultdict(int)
    for iid, info in tqdm(meta_infos.items()):
        for cates in info['categories']:
            for cate in cates[1:]: # 把主类删除 没有用
                attributes[cate] +=1
        try:
            attributes[info['brand']] += 1
        except:
            pass

    print(f'before delete, attribute num:{len(attributes)}')
    new_meta = {}
    for iid, info in tqdm(meta_infos.items()):
        new_meta[iid] = []

        try:
            if attributes[info['brand']] >= attribute_core:
                new_meta[iid].append(info['brand'])
        except:
            pass
        for cates in info['categories']:
            for cate in cates[1:]:
                if attributes[cate] >= attribute_core:
                    new_meta[iid].append(cate)
    # 做映射
    attribute2id = {}
    id2attribute = {}
    attributeid2num = defaultdict(int)
    attribute_id = 1
    items2attributes = {}
    attribute_lens = []

    for iid, attributes in new_meta.items():
        item_id = datamaps['item2id'][iid]
        items2attributes[item_id] = []
        for attribute in attributes:
            if attribute not in attribute2id:
                attribute2id[attribute] = attribute_id
                id2attribute[attribute_id] = attribute
                attribute_id += 1
            attributeid2num[attribute2id[attribute]] += 1
            items2attributes[item_id].append(attribute2id[attribute])
        attribute_lens.append(len(items2attributes[item_id]))
    print(f'before delete, attribute num:{len(attribute2id)}')
    print(f'attributes len, Min:{np.min(attribute_lens)}, Max:{np.max(attribute_lens)}, Avg.:{np.mean(attribute_lens):.4f}')
    # 更新datamap
    datamaps['attribute2id'] = attribute2id
    datamaps['id2attribute'] = id2attribute
    datamaps['attributeid2num'] = attributeid2num
    return len(attribute2id), np.mean(attribute_lens), datamaps, items2attributes


def get_interaction(datas):
    user_seq = {}
    for data in datas:
        user, item, rev, time = data
        if user in user_seq:
            user_seq[user].append((item, rev, time))
        else:
            user_seq[user] = []
            user_seq[user].append((item, rev, time))

    for user, item_time in user_seq.items():
        item_time.sort(key=lambda x: x[2])  # 对各个数据集得单独排序
        items = []
        for t in item_time:
            items.append((t[0],t[1]))
        user_seq[user] = items
    return user_seq

# K-core user_core item_core
def check_Kcore(user_items, user_core, item_core):
    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item[0]] += 1

    for user, num in user_count.items():
        if num < user_core:
            return user_count, item_count, False
    for item, num in item_count.items():
        if num < item_core:
            return user_count, item_count, False
    return user_count, item_count, True # 已经保证Kcore

# 循环过滤 K-core
def filter_Kcore(user_items, user_core, item_core): # user 接所有items
    user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    while not isKcore:
        for user, num in user_count.items():
            if user_count[user] < user_core: # 直接把user 删除
                user_items.pop(user)
            else:
                for item in user_items[user]:
                    if item_count[[item[0]]] < item_core:
                        user_items[user].remove(item)
        user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    return user_items

def id_map(user_items): # user_items dict
    user2id = {} # raw 2 uid
    item2id = {} # raw 2 iid
    id2user = {} # uid 2 raw
    id2item = {} # iid 2 raw
    user_id = 0
    item_id = 0
    final_data = {}
    random_user_list = list(user_items.keys())
    random.shuffle(random_user_list)
    for user in random_user_list:
        items = user_items[user]
        if user not in user2id:
            user2id[user] = str(user_id)
            id2user[str(user_id)] = user
            user_id += 1
        iids = [] # item id lists
        rev_ls = []
        for item in items:
            if item[0] not in item2id:
                item2id[item[0]] = str(item_id)
                id2item[str(item_id)] = item[0]
                item_id += 1
            iids.append(item2id[item[0]])
            rev_ls.append(item[1])
        uid = user2id[user]
        final_data[uid] = (iids,rev_ls)
    data_maps = {
        'user2id': user2id,
        'item2id': item2id,
        'id2user': id2user,
        'id2item': id2item
    }
    return final_data, user_id-1, item_id-1, data_maps

In [3]:
def main(data_name, acronym, data_type='Amazon'):
      assert data_type in {'Amazon', 'Yelp'}
      rating_score = 0.0  # rating score smaller than this score would be deleted
      # user 5-core item 5-core
      user_core = 5
      item_core = 5
      attribute_core = 0

      datas = Amazon(data_name, rating_score=rating_score)

      user_items = get_interaction(datas)
      print(f'{data_name} Raw data has been processed! Lower than {rating_score} are deleted!')
      # raw_id user: [item1, item2, item3...]
      user_items = filter_Kcore(user_items, user_core=user_core, item_core=item_core)
      print(f'User {user_core}-core complete! Item {item_core}-core complete!')

      
      user_count, item_count, _ = check_Kcore(user_items, user_core=user_core, item_core=item_core)
      user_items, user_num, item_num, data_maps = id_map(user_items)
      user_count_list = list(user_count.values())
      user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
      item_count_list = list(item_count.values())
      item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
      interact_num = np.sum([x for x in user_count_list])
      sparsity = (1 - interact_num / (user_num * item_num)) * 100
      show_info = f'Total User: {user_num}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
                  f'Total Item: {item_num}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' + \
                  f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
      print(show_info)


      print('Begin extracting meta infos...')
      print(user_items['0'])
      meta_infos = Amazon_meta(data_name, data_maps)
      attribute_num, avg_attribute, datamaps, item2attributes = get_attribute_Amazon(meta_infos, data_maps, attribute_core)

      print(f'{data_name} & {add_comma(user_num)}& {add_comma(item_num)} & {user_avg:.1f}'
            f'& {item_avg:.1f}& {add_comma(interact_num)}& {sparsity:.2f}\%&{add_comma(attribute_num)}&'
            f'{avg_attribute:.1f} \\')

      return user_items, item2attributes, datamaps
 

In [4]:
# full_data_name = 'Beauty'
# short_data_name = 'beauty'
# meta_file = '../data/Beauty/meta_' + "Beauty" + '.json.gz'

# full_data_name = 'Sports_and_Outdoors'
# short_data_name = 'sport'
# meta_file = '../data/Sports_and_Outdoors/meta_' + "Sports_and_Outdoors" + '.json.gz'

# full_data_name = 'Toys_and_Games'
# short_data_name = 'toys'
# meta_file = '../data/Toys_and_Games/meta_' + "Toys_and_Games" + '.json.gz'


full_data_name = 'Musical_Instruments'
short_data_name = 'sport'
meta_file = '../data/Musical_Instruments/meta_' + "Musical_Instruments" + '.json.gz'

# Sports_and_Outdoors
# Toys_and_Games

data_name = full_data_name
rating_score = 0.0  # rating score smaller than this score would be deleted
# user 5-core item 5-core
user_core = 5
item_core = 5
attribute_core = 0

datas = Amazon(data_name, rating_score=rating_score)

user_items = get_interaction(datas)
print(f'{data_name} Raw data has been processed! Lower than {rating_score} are deleted!')
# raw_id user: [item1, item2, item3...]
user_items = filter_Kcore(user_items, user_core=user_core, item_core=item_core)
print(f'User {user_core}-core complete! Item {item_core}-core complete!')


user_count, item_count, _ = check_Kcore(user_items, user_core=user_core, item_core=item_core)
user_items, user_num, item_num, data_maps = id_map(user_items)
user_count_list = list(user_count.values())
user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
item_count_list = list(item_count.values())
item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
interact_num = np.sum([x for x in user_count_list])
sparsity = (1 - interact_num / (user_num * item_num)) * 100
show_info = f'Total User: {user_num}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
            f'Total Item: {item_num}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' + \
            f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
print(show_info)


print('Begin extracting meta infos...')
meta_infos = Amazon_meta(data_name, data_maps)
attribute_num, avg_attribute, datamaps, item2attributes = get_attribute_Amazon(meta_infos, data_maps, attribute_core)

print(f'{data_name} & {add_comma(user_num)}& {add_comma(item_num)} & {user_avg:.1f}'
    f'& {item_avg:.1f}& {add_comma(interact_num)}& {sparsity:.2f}\%&{add_comma(attribute_num)}&'
    f'{avg_attribute:.1f} \\')

datas = {}

item_asins = list(datamaps['item2id'].keys())
for info in parse(meta_file):
    if info['asin'] not in item_asins:
        continue
    item_id = datamaps['item2id'][info['asin']]
    if 'title' not in info:
        item_title = 'title_' + str(item_id)
    else:
        item_title = info['title']
    if 'description' not in info:
        item_desc = 'description_' + str(item_id)
    else:
        item_desc = info['description']
    datas[item_id] = {
        "title": item_title,
        "description": item_desc
    }

FileNotFoundError: [Errno 2] No such file or directory: '../data/Musical_Instruments/reviews_Musical_Instruments_5.json.gz'

In [17]:

inter_path = "../data/"+full_data_name+"/"+full_data_name+'.inter.json'
item_path = "../data/"+full_data_name+"/"+full_data_name+'.item.json'

with open(inter_path, 'w') as out:
    json.dump(user_items, out, ensure_ascii=False, indent=4)
with open(item_path, 'w') as out:
    json.dump(datas, out, ensure_ascii=False, indent=4)